# PS S6E6 - 011 Blend add010 shared002 LGB check

Experiment: `exp_20260604_011_blend_add010_shared002_lgb_check`

008のAdd007 blend/correlation確認に、010 shared002 LGB as-is を追加するNotebookです。

目的:
- 010を現時点のmainline候補として、007/004/006/003/002とのOOF相関を確認する
- 010単体、010+007、010+004、010+007+004、010+007+004+006 などのblendを確認する
- 010が007を置換するのか、007/004を補助として残す価値があるのかを判断する

重要:
- `.npy` はすべて `/kaggle/input/datasets/mizushimatoshihiko/npy-files` から読み込みます
- 学習は行いません
- original datasetは使いません
- 010 bias searchはこのNotebookでは行いません

In [1]:
import os
import json
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

from scipy.stats import spearmanr
from scipy.optimize import minimize
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression, RidgeClassifier

warnings.filterwarnings("ignore")
try:
    import yaml
except Exception:
    yaml = None

COMPETITION = "PS S6E6 Predicting Stellar Class"
EXP_ID = "exp_20260604_011_blend_add010_shared002_lgb_check"
SEED = 42

TRAIN_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/train.csv")
TEST_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/test.csv")
SAMPLE_SUB_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv")
NPY_DIR = Path("/kaggle/input/datasets/mizushimatoshihiko/npy-files")

OUTDIR = Path("/kaggle/working") / EXP_ID
OUTDIR.mkdir(parents=True, exist_ok=True)

ID_COL = "id"
TARGET_COL = "class"

MODEL_SPECS = [
    {
        "key":"002_lgb_raw",
        "exp_id":"exp_20260601_002_lgb_strict_raw_baseline",
        "family":"LightGBM",
        "role":"raw_reference",
        "oof":"oof_lgb_strict_raw_proba.npy",
        "pred":"pred_lgb_strict_raw_proba.npy",
        "cv":0.9569063401002502,
        "public_lb":0.95800
    },
    {
        "key":"003_cat_raw",
        "exp_id":"exp_20260601_003_cat_strict_raw_baseline",
        "family":"CatBoost",
        "role":"weak_reference",
        "oof":"oof_cat_strict_raw_proba.npy",
        "pred":"pred_cat_strict_raw_proba.npy",
        "cv":0.9524581895303758,
        "public_lb":0.95351
    },
    {
        "key":"004_xgb_raw",
        "exp_id":"exp_20260603_004_xgb_strict_raw_baseline",
        "family":"XGBoost","role":"aux_reference",
        "oof":"oof_xgb_strict_raw_proba.npy",
        "pred":"pred_xgb_strict_raw_proba.npy",
        "cv":0.9557836270283273,
        "public_lb":0.95638
    },
    {
        "key":"006_lgb_color_min",
        "exp_id":"exp_20260603_006_lgb_color_index_min",
        "family":"LightGBM",
        "role":"risky_cv_only_candidate",
        "oof":"oof_lgb_color_index_min_proba.npy",
        "pred":"pred_lgb_color_index_min_proba.npy",
        "cv":0.9570463209115138,
        "public_lb":0.95663
    },
    {
        "key":"007_lgb002_bias",
        "exp_id":"exp_20260603_007_lgb002_bias_search",
        "family":"LightGBM",
        "role":"old_mainline_candidate",
        "oof":"oof_exp_20260603_007_lgb002_bias_search_proba.npy",
        "pred":"pred_exp_20260603_007_lgb002_bias_search_proba.npy",
        "cv":0.9586098708578418,
        "public_lb":0.95921
    },
    {
        "key":"010_shared002_lgb",
        "exp_id":"exp_20260604_010_shared002_lgb_as_is",
        "family":"LightGBM",
        "role":"current_mainline_candidate",
        "oof":"oof_exp_20260604_010_shared002_lgb_as_is_proba.npy",
        "pred":"pred_exp_20260604_010_shared002_lgb_as_is_proba.npy",
        "cv":0.9625129421154345,
        "public_lb":0.96357
    },
]

BLEND_SETS = {
    "A_008_recheck_without_010": ["002_lgb_raw","003_cat_raw","004_xgb_raw","006_lgb_color_min","007_lgb002_bias"],
    "B_010_only": ["010_shared002_lgb"],
    "C_010_007": ["010_shared002_lgb","007_lgb002_bias"],
    "D_010_004": ["010_shared002_lgb","004_xgb_raw"],
    "E_010_007_004": ["010_shared002_lgb","007_lgb002_bias","004_xgb_raw"],
    "F_010_007_004_006": ["010_shared002_lgb","007_lgb002_bias","004_xgb_raw","006_lgb_color_min"],
    "G_010_007_004_003": ["010_shared002_lgb","007_lgb002_bias","004_xgb_raw","003_cat_raw"],
    "H_all_002_003_004_006_007_010": ["002_lgb_raw","003_cat_raw","004_xgb_raw","006_lgb_color_min","007_lgb002_bias","010_shared002_lgb"],
    "I_010_006": ["010_shared002_lgb","006_lgb_color_min"],
    "J_010_002": ["010_shared002_lgb","002_lgb_raw"],
}

print("OUTDIR:", OUTDIR)
print("NPY_DIR:", NPY_DIR)

OUTDIR: /kaggle/working/exp_20260604_011_blend_add010_shared002_lgb_check
NPY_DIR: /kaggle/input/datasets/mizushimatoshihiko/npy-files


In [2]:
def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)

def load_npy_from_dataset(filename):
    path = NPY_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing npy file: {path}Edit MODEL_SPECS if the filename differs.")
    return path

def validate_proba(name, arr, n_rows, n_classes):
    assert arr.shape == (n_rows, n_classes), (name, arr.shape, n_rows, n_classes)
    assert np.isfinite(arr).all(), name
    row_sum = arr.sum(axis=1)
    assert np.allclose(row_sum, 1.0, atol=1e-4), (name, row_sum.min(), row_sum.max())
    assert arr.min() >= -1e-7, (name, arr.min())
    assert arr.max() <= 1.0 + 1e-7, (name, arr.max())

def normalize_rows(p, eps=1e-12):
    p = np.clip(p, eps, None)
    return p / p.sum(axis=1, keepdims=True)

def softmax_np(x, axis=1):
    x = x - np.max(x, axis=axis, keepdims=True)
    ex = np.exp(x)
    return ex / np.sum(ex, axis=axis, keepdims=True)

def proba_to_pred(p): return np.argmax(p, axis=1)
def flatten_proba(p): return np.asarray(p, dtype=float).reshape(-1)

def corr_pearson(a, b):
    a = np.asarray(a, dtype=float).reshape(-1); b = np.asarray(b, dtype=float).reshape(-1)
    if np.std(a) == 0 or np.std(b) == 0: return float("nan")
    return float(np.corrcoef(a, b)[0, 1])

def corr_spearman(a, b):
    a = np.asarray(a, dtype=float).reshape(-1); b = np.asarray(b, dtype=float).reshape(-1)
    if np.std(a) == 0 or np.std(b) == 0: return float("nan")
    return float(spearmanr(a, b).correlation)

def class_recalls(y_true, y_pred, class_names):
    out = {}
    for i, cls in enumerate(class_names):
        m = y_true == i
        out[f"recall_{cls}"] = float((y_pred[m] == i).mean()) if m.any() else float("nan")
    return out

def evaluate_proba(y_true, p, class_names):
    pred = proba_to_pred(p)
    res = {"balanced_accuracy": float(balanced_accuracy_score(y_true, pred))}
    res.update(class_recalls(y_true, pred, class_names))
    return res

def rank_normalize_matrix(p):
    out = np.zeros_like(p, dtype=np.float64); n = p.shape[0]
    for j in range(p.shape[1]):
        r = pd.Series(p[:, j]).rank(method="average").to_numpy()
        out[:, j] = (r - 1) / max(n - 1, 1)
    return normalize_rows(out)

def logit_transform(p, eps=1e-15):
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))

def build_meta_features(keys, proba_dict, mode):
    mats = []
    for k in keys:
        p = proba_dict[k]
        if mode == "raw": mats.append(p)
        elif mode == "rank": mats.append(rank_normalize_matrix(p))
        elif mode == "logit": mats.append(logit_transform(p))
        elif mode == "raw_logit": mats += [p, logit_transform(p)]
        elif mode == "raw_rank_logit": mats += [p, rank_normalize_matrix(p), logit_transform(p)]
        else: raise ValueError(mode)
    return np.concatenate(mats, axis=1)

def average_proba(keys, oof_dict, pred_dict=None):
    oof_avg = normalize_rows(np.mean([oof_dict[k] for k in keys], axis=0))
    pred_avg = normalize_rows(np.mean([pred_dict[k] for k in keys], axis=0)) if pred_dict is not None else None
    return oof_avg, pred_avg

def hill_climb_weights(y_true, probas, steps=(0.1,0.05,0.02,0.01,0.005), max_iter=250):
    n = len(probas); w = np.ones(n)/n
    cur = normalize_rows(sum(w[i]*probas[i] for i in range(n)))
    cur_score = balanced_accuracy_score(y_true, cur.argmax(axis=1))
    hist = [{"iter":0,"score":float(cur_score),"weights":w.copy().tolist()}]
    for step in steps:
        improved=True; it=0
        while improved and it<max_iter:
            improved=False; it+=1; best_score=cur_score; best_w=w.copy()
            for i in range(n):
                for direction in [-1,1]:
                    cand_w=w.copy(); cand_w[i]+=direction*step; cand_w=np.clip(cand_w,0,None)
                    if cand_w.sum()<=0: continue
                    cand_w=cand_w/cand_w.sum()
                    cand=normalize_rows(sum(cand_w[j]*probas[j] for j in range(n)))
                    score=balanced_accuracy_score(y_true, cand.argmax(axis=1))
                    if score>best_score+1e-15:
                        best_score=score; best_w=cand_w.copy()
            if best_score>cur_score+1e-15:
                cur_score=best_score; w=best_w; improved=True
                hist.append({"iter":len(hist),"score":float(cur_score),"weights":w.copy().tolist()})
    final=normalize_rows(sum(w[i]*probas[i] for i in range(n)))
    return w, final, float(cur_score), hist

def nm_optimize_weights(y_true, probas, maxiter=1000):
    n=len(probas)
    def unpack(theta):
        e=np.exp(theta-np.max(theta)); return e/e.sum()
    def objective(theta):
        w=unpack(theta); p=normalize_rows(sum(w[i]*probas[i] for i in range(n)))
        return -balanced_accuracy_score(y_true, p.argmax(axis=1))
    res=minimize(objective, np.zeros(n), method="Nelder-Mead", options={"maxiter":maxiter,"xatol":1e-7,"fatol":1e-10})
    w=unpack(res.x); p=normalize_rows(sum(w[i]*probas[i] for i in range(n)))
    score=balanced_accuracy_score(y_true, p.argmax(axis=1))
    return w, p, float(score), res

In [3]:
# ============================================================
# 2. Load data and OOF/pred
# ============================================================

for p in [TRAIN_PATH, TEST_PATH, SAMPLE_SUB_PATH]:
    if not p.exists(): raise FileNotFoundError(p)
if not NPY_DIR.exists(): raise FileNotFoundError(NPY_DIR)

train = pd.read_csv(TRAIN_PATH); test = pd.read_csv(TEST_PATH); sample = pd.read_csv(SAMPLE_SUB_PATH)
le = LabelEncoder(); y = le.fit_transform(train[TARGET_COL].astype(str))
class_names = le.classes_.tolist(); n_classes = len(class_names)
assert class_names == ["GALAXY", "QSO", "STAR"], class_names

oof = {}; pred = {}; resolved_specs = []
for spec in MODEL_SPECS:
    key = spec["key"]
    oof_path = load_npy_from_dataset(spec["oof"]); pred_path = load_npy_from_dataset(spec["pred"])
    oof_arr = np.load(oof_path).astype(np.float32); pred_arr = np.load(pred_path).astype(np.float32)
    validate_proba(f"{key} oof", oof_arr, len(train), n_classes)
    validate_proba(f"{key} pred", pred_arr, len(test), n_classes)
    oof[key] = oof_arr; pred[key] = pred_arr
    row = dict(spec); row.update({"resolved_oof_path":str(oof_path),"resolved_pred_path":str(pred_path),"oof_shape":list(oof_arr.shape),"pred_shape":list(pred_arr.shape)})
    resolved_specs.append(row)
model_keys = [s["key"] for s in MODEL_SPECS]
print("train:", train.shape); print("test :", test.shape); print("class_names:", class_names)
display(pd.DataFrame(resolved_specs))

train: (577347, 12)
test : (247435, 11)
class_names: ['GALAXY', 'QSO', 'STAR']


,key,exp_id,family,role,oof,pred,cv,public_lb,resolved_oof_path,resolved_pred_path,oof_shape,pred_shape
0,002_lgb_raw,exp_20260601_002_lgb_strict_raw_baseline,LightGBM,raw_reference,oof_lgb_strict_raw_proba.npy,pred_lgb_strict_raw_proba.npy,0.956906,0.95800,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
1,003_cat_raw,exp_20260601_003_cat_strict_raw_baseline,CatBoost,weak_reference,oof_cat_strict_raw_proba.npy,pred_cat_strict_raw_proba.npy,0.952458,0.95351,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
2,004_xgb_raw,exp_20260603_004_xgb_strict_raw_baseline,XGBoost,aux_reference,oof_xgb_strict_raw_proba.npy,pred_xgb_strict_raw_proba.npy,0.955784,0.95638,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
3,006_lgb_color_min,exp_20260603_006_lgb_color_index_min,LightGBM,risky_cv_only_candidate,oof_lgb_color_index_min_proba.npy,pred_lgb_color_index_min_proba.npy,0.957046,0.95663,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
4,007_lgb002_bias,exp_20260603_007_lgb002_bias_search,LightGBM,old_mainline_candidate,oof_exp_20260603_007_lgb002_bias_search_proba.npy,pred_exp_20260603_007_lgb002_bias_search_proba...,0.958610,0.95921,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"
5,010_shared002_lgb,exp_20260604_010_shared002_lgb_as_is,LightGBM,current_mainline_candidate,oof_exp_20260604_010_shared002_lgb_as_is_proba...,pred_exp_20260604_010_shared002_lgb_as_is_prob...,0.962513,0.96357,/kaggle/input/datasets/mizushimatoshihiko/npy-...,/kaggle/input/datasets/mizushimatoshihiko/npy-...,"[577347, 3]","[247435, 3]"


In [4]:
# ============================================================
# 3. Individual scores
# ============================================================

rows=[]
for spec in resolved_specs:
    key=spec["key"]; p=oof[key]; y_pred=proba_to_pred(p); score=balanced_accuracy_score(y,y_pred)
    row={"key":key,"exp_id":spec["exp_id"],"family":spec["family"],"role":spec["role"],"cv_from_memo":spec.get("cv",np.nan),"public_lb":spec.get("public_lb",np.nan),"recomputed_oof_cv":float(score),"delta_recomputed_minus_memo":float(score-spec.get("cv",score))}
    row.update(class_recalls(y,y_pred,class_names)); rows.append(row)
individual_df=pd.DataFrame(rows).sort_values("recomputed_oof_cv",ascending=False)
display(individual_df)
individual_df.to_csv(OUTDIR/"individual_scores.csv",index=False)

,key,exp_id,family,role,cv_from_memo,public_lb,recomputed_oof_cv,delta_recomputed_minus_memo,recall_GALAXY,recall_QSO,recall_STAR
5,010_shared002_lgb,exp_20260604_010_shared002_lgb_as_is,LightGBM,current_mainline_candidate,0.962513,0.96357,0.962513,0.0,0.967675,0.967407,0.952456
4,007_lgb002_bias,exp_20260603_007_lgb002_bias_search,LightGBM,old_mainline_candidate,0.958610,0.95921,0.958610,0.0,0.976693,0.966699,0.932438
3,006_lgb_color_min,exp_20260603_006_lgb_color_index_min,LightGBM,risky_cv_only_candidate,0.957046,0.95663,0.957046,0.0,0.978145,0.964872,0.928122
0,002_lgb_raw,exp_20260601_002_lgb_strict_raw_baseline,LightGBM,raw_reference,0.956906,0.95800,0.956906,0.0,0.978431,0.965000,0.927288
2,004_xgb_raw,exp_20260603_004_xgb_strict_raw_baseline,XGBoost,aux_reference,0.955784,0.95638,0.955784,0.0,0.978582,0.963318,0.925451
1,003_cat_raw,exp_20260601_003_cat_strict_raw_baseline,CatBoost,weak_reference,0.952458,0.95351,0.952458,0.0,0.977432,0.963498,0.916445


In [5]:
# ============================================================
# 4. Pairwise OOF correlation / disagreement / error overlap
# ============================================================

pair_rows=[]
pred_labels={k:proba_to_pred(oof[k]) for k in model_keys}; wrong_flags={k:pred_labels[k]!=y for k in model_keys}
for a,b in combinations(model_keys,2):
    pa,pb=oof[a],oof[b]; ya,yb=pred_labels[a],pred_labels[b]
    wrong_a,wrong_b=wrong_flags[a],wrong_flags[b]; both=wrong_a&wrong_b; either=wrong_a|wrong_b
    row={"model_a":a,"model_b":b,"pearson_flat_proba":corr_pearson(flatten_proba(pa),flatten_proba(pb)),"spearman_flat_proba":corr_spearman(flatten_proba(pa),flatten_proba(pb)),"argmax_agreement":float((ya==yb).mean()),"argmax_disagreement":float((ya!=yb).mean()),"wrong_rate_a":float(wrong_a.mean()),"wrong_rate_b":float(wrong_b.mean()),"both_wrong_rate":float(both.mean()),"either_wrong_rate":float(either.mean()),"error_jaccard":float(both.sum()/max(either.sum(),1)),"a_wrong_b_correct_rate":float((wrong_a&~wrong_b).mean()),"a_correct_b_wrong_rate":float((~wrong_a&wrong_b).mean())}
    for ci,cls in enumerate(class_names):
        row[f"pearson_proba_{cls}"]=corr_pearson(pa[:,ci],pb[:,ci]); row[f"spearman_proba_{cls}"]=corr_spearman(pa[:,ci],pb[:,ci])
    pair_rows.append(row)
pairwise_df=pd.DataFrame(pair_rows).sort_values("pearson_flat_proba")
display(pairwise_df)
pairwise_df.to_csv(OUTDIR/"pairwise_oof_correlation.csv",index=False)

,model_a,model_b,pearson_flat_proba,spearman_flat_proba,argmax_agreement,argmax_disagreement,wrong_rate_a,wrong_rate_b,both_wrong_rate,either_wrong_rate,error_jaccard,a_wrong_b_correct_rate,a_correct_b_wrong_rate,pearson_proba_GALAXY,spearman_proba_GALAXY,pearson_proba_QSO,spearman_proba_QSO,pearson_proba_STAR,spearman_proba_STAR
8,003_cat_raw,010_shared002_lgb,0.984122,0.963576,0.974947,0.025053,0.034134,0.034560,0.022018,0.046676,0.471723,0.012116,0.012542,0.979332,0.953033,0.990859,0.859204,0.966223,0.888231
13,006_lgb_color_min,010_shared002_lgb,0.985263,0.970892,0.976470,0.023530,0.031716,0.034560,0.021575,0.044701,0.482641,0.010141,0.012985,0.980378,0.957714,0.991498,0.865871,0.969025,0.923995
11,004_xgb_raw,010_shared002_lgb,0.985468,0.962224,0.976527,0.023473,0.032128,0.034560,0.021824,0.044864,0.486449,0.010304,0.012736,0.981403,0.952266,0.990590,0.855955,0.969588,0.849454
4,002_lgb_raw,010_shared002_lgb,0.986522,0.971515,0.977527,0.022473,0.031622,0.034560,0.022047,0.044135,0.499549,0.009575,0.012512,0.982032,0.959982,0.992098,0.871675,0.972117,0.918931
14,007_lgb002_bias,010_shared002_lgb,0.987218,0.972312,0.978180,0.021820,0.031676,0.034560,0.022396,0.043840,0.510845,0.009280,0.012164,0.982911,0.959981,0.992502,0.872314,0.973169,0.922695
7,003_cat_raw,007_lgb002_bias,0.996401,0.981468,0.989230,0.010770,0.034134,0.031676,0.027607,0.038202,0.722661,0.006526,0.004069,0.995818,0.982473,0.996796,0.939826,0.992080,0.930138
6,003_cat_raw,006_lgb_color_min,0.996427,0.980368,0.989350,0.010650,0.034134,0.031716,0.027687,0.038162,0.725503,0.006447,0.004029,0.995812,0.981501,0.996792,0.931691,0.991897,0.923943
0,002_lgb_raw,003_cat_raw,0.996519,0.981796,0.989538,0.010462,0.031622,0.034134,0.027735,0.038020,0.729488,0.003887,0.006398,0.995920,0.982472,0.996843,0.939371,0.992158,0.928582
9,004_xgb_raw,006_lgb_color_min,0.997113,0.984696,0.990695,0.009305,0.032128,0.031716,0.027368,0.036475,0.750321,0.004760,0.004347,0.996786,0.984501,0.997045,0.942101,0.993628,0.926341
5,003_cat_raw,004_xgb_raw,0.997419,0.981125,0.990425,0.009575,0.034134,0.032128,0.028434,0.037828,0.751648,0.005700,0.003694,0.996561,0.982859,0.998087,0.946916,0.994371,0.904603


In [6]:
# ============================================================
# 5. Correlation matrices
# ============================================================

pearson_flat=pd.DataFrame(index=model_keys,columns=model_keys,dtype=float)
spearman_flat=pd.DataFrame(index=model_keys,columns=model_keys,dtype=float)
argmax_agreement=pd.DataFrame(index=model_keys,columns=model_keys,dtype=float)
error_jaccard=pd.DataFrame(index=model_keys,columns=model_keys,dtype=float)
for a in model_keys:
    for b in model_keys:
        pearson_flat.loc[a,b]=corr_pearson(flatten_proba(oof[a]),flatten_proba(oof[b]))
        spearman_flat.loc[a,b]=corr_spearman(flatten_proba(oof[a]),flatten_proba(oof[b]))
        argmax_agreement.loc[a,b]=float((pred_labels[a]==pred_labels[b]).mean())
        both=wrong_flags[a]&wrong_flags[b]; either=wrong_flags[a]|wrong_flags[b]
        error_jaccard.loc[a,b]=float(both.sum()/max(either.sum(),1))
display(pearson_flat); display(spearman_flat); display(argmax_agreement); display(error_jaccard)
pearson_flat.to_csv(OUTDIR/"corr_matrix_pearson_flat_proba.csv")
spearman_flat.to_csv(OUTDIR/"corr_matrix_spearman_flat_proba.csv")
argmax_agreement.to_csv(OUTDIR/"matrix_argmax_agreement.csv")
error_jaccard.to_csv(OUTDIR/"matrix_error_jaccard.csv")

,002_lgb_raw,003_cat_raw,004_xgb_raw,006_lgb_color_min,007_lgb002_bias,010_shared002_lgb
002_lgb_raw,1.000000,0.996519,0.997944,0.998256,0.999904,0.986522
003_cat_raw,0.996519,1.000000,0.997419,0.996427,0.996401,0.984122
004_xgb_raw,0.997944,0.997419,1.000000,0.997113,0.997839,0.985468
006_lgb_color_min,0.998256,0.996427,0.997113,1.000000,0.998172,0.985263
007_lgb002_bias,0.999904,0.996401,0.997839,0.998172,1.000000,0.987218
010_shared002_lgb,0.986522,0.984122,0.985468,0.985263,0.987218,1.000000


,002_lgb_raw,003_cat_raw,004_xgb_raw,006_lgb_color_min,007_lgb002_bias,010_shared002_lgb
002_lgb_raw,1.000000,0.981796,0.989543,0.993739,0.999807,0.971515
003_cat_raw,0.981796,1.000000,0.981125,0.980368,0.981468,0.963576
004_xgb_raw,0.989543,0.981125,1.000000,0.984696,0.989643,0.962224
006_lgb_color_min,0.993739,0.980368,0.984696,1.000000,0.993331,0.970892
007_lgb002_bias,0.999807,0.981468,0.989643,0.993331,1.000000,0.972312
010_shared002_lgb,0.971515,0.963576,0.962224,0.970892,0.972312,1.000000


,002_lgb_raw,003_cat_raw,004_xgb_raw,006_lgb_color_min,007_lgb002_bias,010_shared002_lgb
002_lgb_raw,1.000000,0.989538,0.992682,0.992457,0.997759,0.977527
003_cat_raw,0.989538,1.000000,0.990425,0.989350,0.989230,0.974947
004_xgb_raw,0.992682,0.990425,1.000000,0.990695,0.992239,0.976527
006_lgb_color_min,0.992457,0.989350,0.990695,1.000000,0.992232,0.976470
007_lgb002_bias,0.997759,0.989230,0.992239,0.992232,1.000000,0.978180
010_shared002_lgb,0.977527,0.974947,0.976527,0.976470,0.978180,1.000000


,002_lgb_raw,003_cat_raw,004_xgb_raw,006_lgb_color_min,007_lgb002_bias,010_shared002_lgb
002_lgb_raw,1.000000,0.729488,0.798222,0.790618,0.932269,0.499549
003_cat_raw,0.729488,1.000000,0.751648,0.725503,0.722661,0.471723
004_xgb_raw,0.798222,0.751648,1.000000,0.750321,0.787076,0.486449
006_lgb_color_min,0.790618,0.725503,0.750321,1.000000,0.785056,0.482641
007_lgb002_bias,0.932269,0.722661,0.787076,0.785056,1.000000,0.510845
010_shared002_lgb,0.499549,0.471723,0.486449,0.482641,0.510845,1.000000


In [7]:
# ============================================================
# 6. Blend diagnostics
# ============================================================

blend_rows=[]; blend_pred_outputs={}
def record_blend(set_name,method,keys,oof_p,test_p=None,weights=None,extra=None):
    ev=evaluate_proba(y,oof_p,class_names)
    row={"set_name":set_name,"method":method,"keys":",".join(keys),"n_models":len(keys),"balanced_accuracy":ev["balanced_accuracy"],"weights_json":json.dumps({k:float(w) for k,w in zip(keys,weights)},ensure_ascii=False) if weights is not None else None}
    row.update({k:v for k,v in ev.items() if k!="balanced_accuracy"})
    if extra: row.update(extra)
    blend_rows.append(row)
    if test_p is not None: blend_pred_outputs[(set_name,method)]=test_p

for set_name,keys in BLEND_SETS.items():
    print(f"===== {set_name}: {keys} =====")
    probas=[oof[k] for k in keys]; pred_probas=[pred[k] for k in keys]
    avg_oof,avg_pred=average_proba(keys,oof,pred)
    record_blend(set_name,"avg",keys,avg_oof,avg_pred,weights=np.ones(len(keys))/len(keys))
    if len(keys)>=2:
        hc_w,hc_oof,hc_score,hc_hist=hill_climb_weights(y,probas)
        hc_pred=normalize_rows(sum(hc_w[i]*pred_probas[i] for i in range(len(keys))))
        record_blend(set_name,"hc_nonnegative_raw",keys,hc_oof,hc_pred,weights=hc_w,extra={"hc_history_len":len(hc_hist)})
        nm_w,nm_oof,nm_score,nm_res=nm_optimize_weights(y,probas)
        nm_pred=normalize_rows(sum(nm_w[i]*pred_probas[i] for i in range(len(keys))))
        record_blend(set_name,"nm_softmax_raw",keys,nm_oof,nm_pred,weights=nm_w,extra={"nm_success":bool(nm_res.success),"nm_message":str(nm_res.message)})
    for mode in ["raw","raw_logit","raw_rank_logit","logit"]:
        X_meta=build_meta_features(keys,oof,mode); X_test_meta=build_meta_features(keys,pred,mode)
        try:
            lr=LogisticRegression(C=0.05,penalty="l2",solver="lbfgs",max_iter=2000,random_state=SEED,n_jobs=-1)
            lr.fit(X_meta,y); lr_oof=lr.predict_proba(X_meta); lr_pred=lr.predict_proba(X_test_meta)
            record_blend(set_name,f"logreg_{mode}",keys,lr_oof,lr_pred,extra={"C":0.05})
        except Exception as e: print(f"[WARN] logreg failed: {set_name} {mode}: {e}")
        try:
            rc=RidgeClassifier(alpha=10.0,random_state=SEED)
            rc.fit(X_meta,y); rc_oof=softmax_np(rc.decision_function(X_meta),axis=1); rc_pred=softmax_np(rc.decision_function(X_test_meta),axis=1)
            record_blend(set_name,f"ridgeclf_{mode}",keys,rc_oof,rc_pred,extra={"alpha":10.0})
        except Exception as e: print(f"[WARN] ridgeclf failed: {set_name} {mode}: {e}")
blend_df=pd.DataFrame(blend_rows).sort_values("balanced_accuracy",ascending=False)
display(blend_df.head(100))
blend_df.to_csv(OUTDIR/"blend_diagnostics.csv",index=False)

===== A_008_recheck_without_010: ['002_lgb_raw', '003_cat_raw', '004_xgb_raw', '006_lgb_color_min', '007_lgb002_bias'] =====
[WARN] logreg failed: A_008_recheck_without_010 raw_logit: Input X contains infinity or a value too large for dtype('float64').
[WARN] ridgeclf failed: A_008_recheck_without_010 raw_logit: Input X contains infinity or a value too large for dtype('float32').
[WARN] logreg failed: A_008_recheck_without_010 raw_rank_logit: Input X contains infinity or a value too large for dtype('float64').
[WARN] ridgeclf failed: A_008_recheck_without_010 raw_rank_logit: Input X contains infinity or a value too large for dtype('float64').
[WARN] logreg failed: A_008_recheck_without_010 logit: Input X contains infinity or a value too large for dtype('float64').
[WARN] ridgeclf failed: A_008_recheck_without_010 logit: Input X contains infinity or a value too large for dtype('float32').
===== B_010_only: ['010_shared002_lgb'] =====
===== C_010_007: ['010_shared002_lgb', '007_lgb002_bi

,set_name,method,keys,n_models,balanced_accuracy,weights_json,recall_GALAXY,recall_QSO,recall_STAR,hc_history_len,nm_success,nm_message,C,alpha
36,F_010_007_004_006,hc_nonnegative_raw,"010_shared002_lgb,007_lgb002_bias,004_xgb_raw,...",4,0.964055,"{""010_shared002_lgb"": 0.548400250123365, ""007_...",0.973853,0.967740,0.950571,12.0,NaN,NaN,NaN,NaN
46,H_all_002_003_004_006_007_010,hc_nonnegative_raw,"002_lgb_raw,003_cat_raw,004_xgb_raw,006_lgb_co...",6,0.964033,"{""002_lgb_raw"": 0.0, ""003_cat_raw"": 0.0, ""004_...",0.973617,0.967706,0.950776,14.0,NaN,NaN,NaN,NaN
15,C_010_007,hc_nonnegative_raw,"010_shared002_lgb,007_lgb002_bias",2,0.964032,"{""010_shared002_lgb"": 0.5454545454545454, ""007...",0.973538,0.967928,0.950631,2.0,NaN,NaN,NaN,NaN
31,E_010_007_004,hc_nonnegative_raw,"010_shared002_lgb,007_lgb002_bias,004_xgb_raw",3,0.963989,"{""010_shared002_lgb"": 0.547593042200316, ""007_...",0.973628,0.967877,0.950462,10.0,NaN,NaN,NaN,NaN
41,G_010_007_004_003,hc_nonnegative_raw,"010_shared002_lgb,007_lgb002_bias,004_xgb_raw,...",4,0.963971,"{""010_shared002_lgb"": 0.5634989830138885, ""007...",0.973434,0.967800,0.950679,11.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3,A_008_recheck_without_010,logreg_raw,"002_lgb_raw,003_cat_raw,004_xgb_raw,006_lgb_co...",5,0.955794,None,0.980338,0.963370,0.923674,NaN,NaN,NaN,0.05,NaN
60,I_010_006,ridgeclf_logit,"010_shared002_lgb,006_lgb_color_min",2,0.952574,None,0.979257,0.951128,0.927337,NaN,NaN,NaN,NaN,10.0
24,C_010_007,ridgeclf_logit,"010_shared002_lgb,007_lgb002_bias",2,0.951361,None,0.979029,0.949276,0.925777,NaN,NaN,NaN,NaN,10.0
13,B_010_only,ridgeclf_logit,010_shared002_lgb,1,0.951259,None,0.978515,0.951188,0.924073,NaN,NaN,NaN,NaN,10.0


In [8]:
# ============================================================
# 7. Save top blend submissions
# ============================================================

save_methods_priority={"avg","hc_nonnegative_raw","nm_softmax_raw"}
top_save=blend_df[blend_df["method"].isin(save_methods_priority)].head(12).copy()
submission_rows=[]
for _,row in top_save.iterrows():
    set_name=row["set_name"]; method=row["method"]; test_p=blend_pred_outputs.get((set_name,method))
    if test_p is None: continue
    labels=le.inverse_transform(test_p.argmax(axis=1))
    sub=pd.DataFrame({ID_COL:test[ID_COL].values,TARGET_COL:labels})
    assert sub.shape[0]==sample.shape[0]
    assert sub.columns.tolist()==sample.columns.tolist()
    assert sub[ID_COL].equals(sample[ID_COL])
    fname=f"submission_exp_20260604_011_blend_add010_shared002_lgb_check_{set_name}_{method}.csv".replace("/","_")
    sub_path=OUTDIR/fname; sub.to_csv(sub_path,index=False)
    pred_fname=f"pred_exp_20260604_011_blend_add010_shared002_lgb_check_{set_name}_{method}_proba.npy".replace("/","_")
    pred_path=OUTDIR/pred_fname; np.save(pred_path,test_p.astype(np.float32))
    submission_rows.append({"set_name":set_name,"method":method,"balanced_accuracy":row["balanced_accuracy"],"submission":fname,"pred_proba":pred_fname})
submission_df=pd.DataFrame(submission_rows)
display(submission_df)
submission_df.to_csv(OUTDIR/"saved_submission_candidates.csv",index=False)

,set_name,method,balanced_accuracy,submission,pred_proba
0,F_010_007_004_006,hc_nonnegative_raw,0.964055,submission_exp_20260604_011_blend_add010_share...,pred_exp_20260604_011_blend_add010_shared002_l...
1,H_all_002_003_004_006_007_010,hc_nonnegative_raw,0.964033,submission_exp_20260604_011_blend_add010_share...,pred_exp_20260604_011_blend_add010_shared002_l...
2,C_010_007,hc_nonnegative_raw,0.964032,submission_exp_20260604_011_blend_add010_share...,pred_exp_20260604_011_blend_add010_shared002_l...
3,E_010_007_004,hc_nonnegative_raw,0.963989,submission_exp_20260604_011_blend_add010_share...,pred_exp_20260604_011_blend_add010_shared002_l...
4,G_010_007_004_003,hc_nonnegative_raw,0.963971,submission_exp_20260604_011_blend_add010_share...,pred_exp_20260604_011_blend_add010_shared002_l...
5,I_010_006,hc_nonnegative_raw,0.963928,submission_exp_20260604_011_blend_add010_share...,pred_exp_20260604_011_blend_add010_shared002_l...
6,C_010_007,nm_softmax_raw,0.963890,submission_exp_20260604_011_blend_add010_share...,pred_exp_20260604_011_blend_add010_shared002_l...
7,C_010_007,avg,0.963889,submission_exp_20260604_011_blend_add010_share...,pred_exp_20260604_011_blend_add010_shared002_l...
8,D_010_004,hc_nonnegative_raw,0.963780,submission_exp_20260604_011_blend_add010_share...,pred_exp_20260604_011_blend_add010_shared002_l...
9,I_010_006,avg,0.963736,submission_exp_20260604_011_blend_add010_share...,pred_exp_20260604_011_blend_add010_shared002_l...


In [9]:
# ============================================================
# 8. Role judgement
# ============================================================

def get_ind_score(key):
    return float(individual_df.loc[individual_df["key"]==key,"recomputed_oof_cv"].iloc[0])
best_blend=blend_df.iloc[0].to_dict()
judgement={
    "current_single_best":{"key":"010_shared002_lgb","cv":get_ind_score("010_shared002_lgb"),"public_lb":0.96357,"status":"current_mainline_single"},
    "best_blend_oof":best_blend,
    "main_questions":{
        "does_007_add_to_010":"Check C_010_007 and E_010_007_004 against B_010_only.",
        "does_004_add_to_010":"Check D_010_004 and E_010_007_004 against B_010_only.",
        "does_006_add_to_010":"Check I_010_006 and F_010_007_004_006.",
        "should_002_remain":"Check J_010_002 and H_all.",
        "should_003_remain":"Check G/H and HC weights for 003.",
    },
    "initial_policy":{"010_shared002_lgb":"mainline_candidate","007_lgb002_bias":"old_mainline_candidate_or_aux","004_xgb_raw":"auxiliary_candidate","006_lgb_color_min":"hold_risky_cv_only","002_lgb_raw":"raw_reference","003_cat_raw":"weak_reference"},
}
save_json(judgement,OUTDIR/"role_judgement.json")
print(json.dumps(judgement,indent=2,ensure_ascii=False))

{
  "current_single_best": {
    "key": "010_shared002_lgb",
    "cv": 0.9625129421154345,
    "public_lb": 0.96357,
    "status": "current_mainline_single"
  },
  "best_blend_oof": {
    "set_name": "F_010_007_004_006",
    "method": "hc_nonnegative_raw",
    "keys": "010_shared002_lgb,007_lgb002_bias,004_xgb_raw,006_lgb_color_min",
    "n_models": 4,
    "balanced_accuracy": 0.9640545915420509,
    "weights_json": "{\"010_shared002_lgb\": 0.548400250123365, \"007_lgb002_bias\": 0.21337268183885552, \"004_xgb_raw\": 0.0, \"006_lgb_color_min\": 0.23822706803777954}",
    "recall_GALAXY": 0.9738529193599661,
    "recall_QSO": 0.967740283243557,
    "recall_STAR": 0.9505705720226295,
    "hc_history_len": 12.0,
    "nm_success": NaN,
    "nm_message": NaN,
    "C": NaN,
    "alpha": NaN
  },
  "main_questions": {
    "does_007_add_to_010": "Check C_010_007 and E_010_007_004 against B_010_only.",
    "does_004_add_to_010": "Check D_010_004 and E_010_007_004 against B_010_only.",
    "does

In [10]:
# ============================================================
# 9. Save summary / memo
# ============================================================

summary={"competition":COMPETITION,"exp_id":EXP_ID,"status":"completed","purpose":"Add 010 shared002 LGB to 008 blend/correlation check","npy_dir":str(NPY_DIR),"model_keys":model_keys,"class_names":class_names,"resolved_specs":resolved_specs,"individual_scores":individual_df.to_dict(orient="records"),"pairwise_oof_correlation":pairwise_df.to_dict(orient="records"),"blend_diagnostics":blend_df.to_dict(orient="records"),"saved_submission_candidates":submission_df.to_dict(orient="records") if "submission_df" in globals() else [],"role_judgement":judgement}
save_json(summary,OUTDIR/"blend_add010_summary.json")

memo={
    "experiment":{"id":EXP_ID,"title":"Blend check after adding 010 shared002 LGB","competition":COMPETITION,"status":"completed","metric":"balanced_accuracy_score","created_at":"2026-06-04"},
    "objective":{"summary":"Load OOF/pred probabilities from the fixed npy-files Kaggle dataset folder, add 010 shared002 LGB output to the 008 blend/correlation table, and check whether 007/004/006/002/003 add value after 010.","success_criteria":["load 002/003/004/006/007/010 OOF and pred npy files from /kaggle/input/datasets/mizushimatoshihiko/npy-files","recompute individual balanced accuracy","compute pairwise OOF correlations","evaluate predefined blend sets centered on 010","save top blend submission candidates","save memo and summary"]},
    "inputs":{"npy_dir":str(NPY_DIR),"models":resolved_specs,"blend_sets":BLEND_SETS},
    "outputs":{"individual_scores":"individual_scores.csv","pairwise_oof_correlation":"pairwise_oof_correlation.csv","corr_matrix_pearson_flat_proba":"corr_matrix_pearson_flat_proba.csv","corr_matrix_spearman_flat_proba":"corr_matrix_spearman_flat_proba.csv","matrix_argmax_agreement":"matrix_argmax_agreement.csv","matrix_error_jaccard":"matrix_error_jaccard.csv","blend_diagnostics":"blend_diagnostics.csv","saved_submission_candidates":"saved_submission_candidates.csv","role_judgement":"role_judgement.json","summary":"blend_add010_summary.json"},
    "judgement":{"status":"completed_pending_manual_review","initial_view":"010 is the current mainline single candidate. 011 should decide whether 007, 004, or other older candidates add value after 010.","next_action":["Review blend_diagnostics.csv","Review 010 vs 007 / 004 / 006 correlations","If a blend candidate meaningfully beats 010 single CV, submit one candidate","If not, keep 010 as mainline and run bias search on 010 next"]},
}

memo_path=OUTDIR/"memo.yml"
if yaml is not None:
    with open(memo_path,"w",encoding="utf-8") as f: yaml.safe_dump(memo,f,allow_unicode=True,sort_keys=False)
else:
    with open(memo_path,"w",encoding="utf-8") as f: f.write(json.dumps(memo,indent=2,ensure_ascii=False,default=str))
print("Saved files:")
for p in sorted(OUTDIR.glob("*")): print(" -",p.name)

Saved files:
 - blend_add010_summary.json
 - blend_diagnostics.csv
 - corr_matrix_pearson_flat_proba.csv
 - corr_matrix_spearman_flat_proba.csv
 - individual_scores.csv
 - matrix_argmax_agreement.csv
 - matrix_error_jaccard.csv
 - memo.yml
 - pairwise_oof_correlation.csv
 - pred_exp_20260604_011_blend_add010_shared002_lgb_check_C_010_007_avg_proba.npy
 - pred_exp_20260604_011_blend_add010_shared002_lgb_check_C_010_007_hc_nonnegative_raw_proba.npy
 - pred_exp_20260604_011_blend_add010_shared002_lgb_check_C_010_007_nm_softmax_raw_proba.npy
 - pred_exp_20260604_011_blend_add010_shared002_lgb_check_D_010_004_hc_nonnegative_raw_proba.npy
 - pred_exp_20260604_011_blend_add010_shared002_lgb_check_E_010_007_004_hc_nonnegative_raw_proba.npy
 - pred_exp_20260604_011_blend_add010_shared002_lgb_check_F_010_007_004_006_hc_nonnegative_raw_proba.npy
 - pred_exp_20260604_011_blend_add010_shared002_lgb_check_G_010_007_004_003_hc_nonnegative_raw_proba.npy
 - pred_exp_20260604_011_blend_add010_shared002_